# Previsão de xG e Resultados — Google Colab

Notebook completo para:
1. Upload dos arquivos (`passadas.xlsx`, `atuais.xlsx`, `proxima.xlsx`, `modelo_v2.py`)
2. Treino/execução do pipeline
3. Previsão da próxima rodada
4. Ranking de probabilidades (1X2, Over/Under e placar mais provável)


In [ ]:
!pip -q install pandas numpy scipy scikit-learn openpyxl


In [ ]:
import os
import pandas as pd
import numpy as np
from scipy.stats import poisson
from google.colab import files

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)


In [ ]:
print('Faça upload de: passadas.xlsx, atuais.xlsx, proxima.xlsx e modelo_v2.py')
uploaded = files.upload()
print('Arquivos enviados:', list(uploaded.keys()))


In [ ]:
required = ['passadas.xlsx', 'atuais.xlsx', 'proxima.xlsx', 'modelo_v2.py']
missing = [f for f in required if not os.path.exists(f)]
if missing:
    raise FileNotFoundError(f'Arquivos ausentes: {missing}')
print('✅ Tudo certo. Arquivos obrigatórios encontrados.')


In [ ]:
!python modelo_v2.py


## Carregar saídas do modelo


In [ ]:
out = 'previsao_v2.xlsx'
xls = pd.ExcelFile(out)
print('Abas geradas:', xls.sheet_names)

pred = pd.read_excel(out, sheet_name='previsoes')
display(pred.head(20))


## Probabilidades 1X2 + placar mais provável

Usa as lambdas finais (`lambda_home`, `lambda_away`) e distribuição de Poisson independente
para calcular:
- Vitória mandante (1)
- Empate (X)
- Vitória visitante (2)
- Placar exato mais provável


In [ ]:
def score_matrix(lh, la, max_goals=10):
    h = poisson.pmf(np.arange(max_goals+1), lh)
    a = poisson.pmf(np.arange(max_goals+1), la)
    m = np.outer(h, a)

    # Ajusta pequena massa de cauda para manter soma ~1
    m = m / m.sum()
    return m

rows = []
for _, r in pred.iterrows():
    lh = float(r['lambda_home'])
    la = float(r['lambda_away'])
    m = score_matrix(lh, la, max_goals=10)

    p_home = np.tril(m, -1).sum()  # hg > ag
    p_draw = np.trace(m)
    p_away = np.triu(m, 1).sum()   # hg < ag

    idx = np.unravel_index(np.argmax(m), m.shape)
    score = f"{idx[0]}-{idx[1]}"

    rows.append({
        'home': r['home'],
        'away': r['away'],
        'P1_home_win': p_home,
        'PX_draw': p_draw,
        'P2_away_win': p_away,
        'placar_mais_provavel': score,
        'prob_placar': m[idx]
    })

p1x2 = pd.DataFrame(rows)
for c in ['P1_home_win','PX_draw','P2_away_win','prob_placar']:
    p1x2[c] = (p1x2[c]*100).round(2)

display(p1x2)


## Over/Under de gols totais (0.5, 1.5, 2.5, 3.5)


In [ ]:
def total_over_probs(lh, la, max_goals=12):
    m = score_matrix(lh, la, max_goals=max_goals)
    probs = {}
    for line in [0.5, 1.5, 2.5, 3.5]:
        k = int(line)
        # total goals > k
        p_over = sum(m[i,j] for i in range(m.shape[0]) for j in range(m.shape[1]) if (i+j) > k)
        probs[f'over_{str(line).replace(".","_")}'] = p_over
    return probs

ou_rows = []
for _, r in pred.iterrows():
    lh = float(r['lambda_home'])
    la = float(r['lambda_away'])
    probs = total_over_probs(lh, la)
    ou_rows.append({'home': r['home'], 'away': r['away'], **probs})

ou = pd.DataFrame(ou_rows)
for c in [c for c in ou.columns if c.startswith('over_')]:
    ou[c] = (ou[c]*100).round(2)

display(ou)


## Ranking final de confiança (sinal mais forte por jogo)


In [ ]:
rank = p1x2.merge(ou, on=['home','away'])

rank['melhor_mercado'] = rank[['P1_home_win','PX_draw','P2_away_win','over_0_5','over_1_5','over_2_5','over_3_5']].idxmax(axis=1)
rank['confianca_%'] = rank[['P1_home_win','PX_draw','P2_away_win','over_0_5','over_1_5','over_2_5','over_3_5']].max(axis=1)

rank = rank.sort_values('confianca_%', ascending=False).reset_index(drop=True)
display(rank)


In [ ]:
with pd.ExcelWriter('previsao_colab_analise.xlsx', engine='openpyxl') as writer:
    pred.to_excel(writer, sheet_name='previsoes_modelo', index=False)
    p1x2.to_excel(writer, sheet_name='probabilidades_1x2', index=False)
    ou.to_excel(writer, sheet_name='over_under_totais', index=False)
    rank.to_excel(writer, sheet_name='ranking_final', index=False)

print('✅ Arquivo gerado: previsao_colab_analise.xlsx')
files.download('previsao_colab_analise.xlsx')
